# NetrAI — DR Screening Training Pipeline
**SIH 2026 | Problem Statement 26038 | Team Biscuit**

Training EfficientNet-B4 hybrid model on APTOS 2019 dataset.
Target: QWK > 0.85 | Referable Sensitivity > 90%

In [ ]:
!pip install timm albumentations grad-cam -q

In [ ]:
import os, gc, cv2, timm, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import seaborn as sns
from tqdm import tqdm
warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Configuration
CFG = {
    'train_csv'    : '/kaggle/input/aptos2019-blindness-detection/train.csv',
    'train_img_dir': '/kaggle/input/aptos2019-blindness-detection/train_images',
    'output_dir'   : '/kaggle/working/',
    'model_name'   : 'efficientnet_b4',
    'img_size'     : 512,
    'num_classes'  : 5,
    'n_folds'      : 5,
    'train_folds'  : [0],
    'epochs'       : 30,
    'batch_size'   : 16,
    'num_workers'  : 4,
    'pin_memory'   : True,
    'lr'           : 3e-4,
    'weight_decay' : 1e-2,
    'min_lr'       : 1e-6,
    'focal_gamma'  : 2.0,
    'label_smooth' : 0.1,
    'tta_steps'    : 5,
    'seed'         : 42,
    'use_amp'      : True,
}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(CFG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# Ben Graham Preprocessing
def ben_graham_preprocess(img, sigmaX=10):
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0, 0), sigmaX), -4, 128)
    return img

def crop_circle(img, tol=7):
    if img.ndim == 2:
        mask = img > tol
    else:
        mask = img.max(axis=2) > tol
    m, n = mask.shape
    mask0, mask1 = mask.any(1), mask.any(0)
    row_min, row_max = mask0.argmax(), m - mask0[::-1].argmax()
    col_min, col_max = mask1.argmax(), n - mask1[::-1].argmax()
    return img[row_min:row_max, col_min:col_max]

def load_and_preprocess(img_path, img_size, apply_ben_graham=True):
    img = cv2.imread(img_path)
    if img is None:
        return np.zeros((img_size, img_size, 3), dtype=np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = crop_circle(img)
    img = cv2.resize(img, (img_size, img_size))
    if apply_ben_graham:
        img = ben_graham_preprocess(img)
    return img

In [ ]:
# Augmentation Pipeline
def get_train_transforms(img_size):
    return A.Compose([
        A.RandomRotate90(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.7),
        A.OneOf([
            A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=1),
            A.GridDistortion(p=1),
            A.OpticalDistortion(distort_limit=2, shift_limit=0.5, p=1),
        ], p=0.3),
        A.OneOf([
            A.GaussNoise(p=1),
            A.Blur(blur_limit=3, p=1),
        ], p=0.3),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
            A.RandomGamma(p=1),
            A.CLAHE(p=1),
        ], p=0.4),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.4),
        A.CoarseDropout(max_holes=8, max_height=img_size//16, max_width=img_size//16,
                        min_holes=5, fill_value=0, p=0.5),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_valid_transforms(img_size):
    return A.Compose([
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

In [ ]:
# Dataset
class APTOSDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None, apply_ben_graham=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms
        self.apply_ben_graham = apply_ben_graham
        self.img_size = CFG['img_size']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['id_code'] + '.png')
        img = load_and_preprocess(img_path, self.img_size, self.apply_ben_graham)
        if self.transforms:
            augmented = self.transforms(image=img)
            img = augmented['image']
        label = int(row['diagnosis'])
        return img, label

In [ ]:
# Model
class NetrAIModel(nn.Module):
    def __init__(self, model_name='efficientnet_b4', num_classes=5, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained)
        in_features = self.backbone.get_classifier().in_features
        self.backbone.reset_classifier(0)
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.3),
        )
        self.grade_head = nn.Linear(256, num_classes)
        self.ref_head = nn.Linear(256, 1)

    def forward(self, x):
        features = self.backbone(x)
        features = self.head(features)
        grade_logits = self.grade_head(features)
        ref_logits = self.ref_head(features).squeeze(1)
        return grade_logits, ref_logits

In [ ]:
# Loss Functions
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        n_classes = logits.size(1)
        if self.label_smoothing > 0:
            smooth_targets = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1.0)
            smooth_targets = smooth_targets * (1 - self.label_smoothing) + self.label_smoothing / n_classes
            log_probs = F.log_softmax(logits, dim=1)
            ce_loss = -(smooth_targets * log_probs).sum(dim=1)
        else:
            ce_loss = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        probs = F.softmax(logits, dim=1)
        p_t = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - p_t) ** self.gamma
        return (focal_weight * ce_loss).mean()

def get_class_weights(labels):
    counts = np.bincount(labels, minlength=5)
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * 5
    return torch.FloatTensor(weights).to(DEVICE)

class CombinedLoss(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__()
        self.focal = FocalLoss(gamma=CFG['focal_gamma'], weight=class_weights, label_smoothing=CFG['label_smooth'])
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, grade_logits, ref_logits, grade_targets):
        focal_loss = self.focal(grade_logits, grade_targets)
        ref_targets = (grade_targets >= 2).float()
        bce_loss = self.bce(ref_logits, ref_targets)
        return 0.8 * focal_loss + 0.2 * bce_loss

In [ ]:
# Metrics
def compute_qwk(preds, targets):
    return cohen_kappa_score(targets, preds, weights='quadratic')

def find_optimal_thresholds(oof_preds, oof_targets):
    from scipy.optimize import minimize
    coef = [0.5, 1.5, 2.5, 3.5]
    def neg_kappa(coef):
        preds = np.digitize(oof_preds, coef).clip(0, 4)
        return -compute_qwk(preds, oof_targets)
    result = minimize(neg_kappa, coef, method='Nelder-Mead')
    return result.x

def apply_thresholds(preds_continuous, thresholds):
    return np.digitize(preds_continuous, sorted(thresholds)).clip(0, 4)

def compute_referable_metrics(preds, targets):
    ref_preds = (np.array(preds) >= 2).astype(int)
    ref_targets = (np.array(targets) >= 2).astype(int)
    tn, fp, fn, tp = confusion_matrix(ref_targets, ref_preds).ravel()
    sensitivity = tp / (tp + fn + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    return sensitivity, specificity

In [ ]:
# Training & Validation
def train_epoch(model, loader, optimizer, criterion, scaler, epoch):
    model.train()
    losses = []
    pbar = tqdm(loader, desc=f'Train Epoch {epoch}')
    for imgs, labels in pbar:
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast(enabled=CFG['use_amp']):
            grade_logits, ref_logits = model(imgs)
            loss = criterion(grade_logits, ref_logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
        avg_loss = np.mean(losses[-20:])
        pbar.set_postfix(loss=f'{avg_loss:.4f}')
    return np.mean(losses)

@torch.no_grad()
def valid_epoch(model, loader, criterion):
    model.eval()
    losses, all_preds, all_labels = [], [], []
    for imgs, labels in tqdm(loader, desc='Validation'):
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)
        with autocast(enabled=CFG['use_amp']):
            grade_logits, ref_logits = model(imgs)
            loss = criterion(grade_logits, ref_logits, labels)
        losses.append(loss.item())
        probs = F.softmax(grade_logits, dim=1).cpu().numpy()
        cont_preds = (probs * np.arange(5)).sum(axis=1)
        all_preds.extend(cont_preds.tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
    return np.mean(losses), np.array(all_preds), np.array(all_labels)

@torch.no_grad()
def predict_tta(model, loader, n_tta=5):
    model.eval()
    all_probs = []
    for _ in range(n_tta):
        batch_probs = []
        for imgs, _ in loader:
            imgs = imgs.to(DEVICE)
            with autocast(enabled=CFG['use_amp']):
                grade_logits, _ = model(imgs)
            probs = F.softmax(grade_logits, dim=1).cpu().numpy()
            batch_probs.append(probs)
        all_probs.append(np.concatenate(batch_probs))
    avg_probs = np.mean(all_probs, axis=0)
    cont_preds = (avg_probs * np.arange(5)).sum(axis=1)
    return cont_preds, avg_probs

In [ ]:
# Main Training Loop
df = pd.read_csv(CFG['train_csv'])
print(f'Dataset: {len(df)} images')
print(df['diagnosis'].value_counts().sort_index())

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(df, df['diagnosis'])):
    df.loc[val_idx, 'fold'] = fold

oof_preds = np.zeros(len(df))
oof_targets = df['diagnosis'].values.copy()
fold_scores = []

for fold in CFG['train_folds']:
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'  FOLD {fold} / {CFG["n_folds"]-1}')
    print(sep)

    train_df = df[df['fold'] != fold].reset_index(drop=True)
    valid_df = df[df['fold'] == fold].reset_index(drop=True)
    print(f'Train: {len(train_df)} | Val: {len(valid_df)}')

    class_weights = get_class_weights(train_df['diagnosis'].values)
    print(f'Class weights: {class_weights.cpu().numpy().round(3)}')

    train_ds = APTOSDataset(train_df, CFG['train_img_dir'], get_train_transforms(CFG['img_size']))
    valid_ds = APTOSDataset(valid_df, CFG['train_img_dir'], get_valid_transforms(CFG['img_size']))

    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                               num_workers=CFG['num_workers'], pin_memory=CFG['pin_memory'])
    valid_loader = DataLoader(valid_ds, batch_size=CFG['batch_size'] * 2, shuffle=False,
                               num_workers=CFG['num_workers'], pin_memory=CFG['pin_memory'])

    model = NetrAIModel(CFG['model_name'], CFG['num_classes']).to(DEVICE)
    criterion = CombinedLoss(class_weights=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1, eta_min=CFG['min_lr'])
    scaler = GradScaler(enabled=CFG['use_amp'])

    best_qwk = -1.0
    history = {'train_loss': [], 'val_loss': [], 'qwk': []}

    for epoch in range(1, CFG['epochs'] + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, epoch)
        val_loss, val_preds_cont, val_labels = valid_epoch(model, valid_loader, criterion)
        scheduler.step()

        val_preds_discrete = np.round(val_preds_cont).clip(0, 4).astype(int)
        qwk = compute_qwk(val_preds_discrete, val_labels)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['qwk'].append(qwk)

        n_epochs = CFG['epochs']
        cur_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:02d}/{n_epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | QWK: {qwk:.4f} | LR: {cur_lr:.2e}')

        if qwk > best_qwk:
            best_qwk = qwk
            save_path = os.path.join(CFG['output_dir'], f'netrai_fold{fold}_best.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'qwk': qwk,
                'fold': fold,
                'cfg': CFG,
            }, save_path)
            print(f'  >> Saved best model (QWK={qwk:.4f})')

    # Post-fold: Optimize thresholds
    print('\nOptimizing classification thresholds...')
    ckpt_path = os.path.join(CFG['output_dir'], f'netrai_fold{fold}_best.pth')
    model.load_state_dict(torch.load(ckpt_path)['model_state_dict'])
    _, oof_cont, oof_labs = valid_epoch(model, valid_loader, criterion)
    opt_thresh = find_optimal_thresholds(oof_cont, oof_labs)
    oof_preds_opt = apply_thresholds(oof_cont, opt_thresh)
    opt_qwk = compute_qwk(oof_preds_opt, oof_labs)

    print(f'Optimal thresholds: {opt_thresh.round(3)}')
    print(f'QWK before opt: {best_qwk:.4f} | After: {opt_qwk:.4f}')

    val_idx = df[df['fold'] == fold].index
    oof_preds[val_idx] = oof_cont
    np.save(os.path.join(CFG['output_dir'], f'fold{fold}_thresholds.npy'), opt_thresh)

    sens, spec = compute_referable_metrics(oof_preds_opt, oof_labs)
    fold_scores.append({'fold': fold, 'qwk': opt_qwk, 'sensitivity': sens, 'specificity': spec})
    print(f'Referable DR — Sensitivity: {sens:.3f} | Specificity: {spec:.3f}')

    del model, train_ds, valid_ds, train_loader, valid_loader
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n{"=" * 60}')
print('TRAINING COMPLETE')
for s in fold_scores:
    print(f"Fold {s['fold']}: QWK={s['qwk']:.4f} | Sens={s['sensitivity']:.3f} | Spec={s['specificity']:.3f}")

In [ ]:
# Generate Validation Plots
from sklearn.metrics import roc_curve, auc

fold = CFG['train_folds'][0]
ckpt = torch.load(os.path.join(CFG['output_dir'], f'netrai_fold{fold}_best.pth'))
thresh = np.load(os.path.join(CFG['output_dir'], f'fold{fold}_thresholds.npy'))

model = NetrAIModel(CFG['model_name'], CFG['num_classes']).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])

val_df = df[df['fold'] == fold].reset_index(drop=True)
val_ds = APTOSDataset(val_df, CFG['train_img_dir'], get_valid_transforms(CFG['img_size']))
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=4)

cont_preds, avg_probs = predict_tta(model, val_loader, n_tta=CFG['tta_steps'])
final_preds = apply_thresholds(cont_preds, thresh)
targets = val_df['diagnosis'].values

final_qwk = compute_qwk(final_preds, targets)
sens, spec = compute_referable_metrics(final_preds, targets)
print(f'Final QWK (TTA + optimized thresh): {final_qwk:.4f}')
print(f'Referable Sensitivity: {sens:.3f} | Specificity: {spec:.3f}')

In [ ]:
# Confusion Matrix & ROC
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='white')

cm = confusion_matrix(targets, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No DR', 'Mild', 'Moderate', 'Severe', 'PDR'],
            yticklabels=['No DR', 'Mild', 'Moderate', 'Severe', 'PDR'])
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('True', fontsize=12)
qwk_val = compute_qwk(final_preds, targets)
axes[0].set_title(f'NetrAI Confusion Matrix | QWK={qwk_val:.4f}', fontsize=13, fontweight='bold')

ref_probs = avg_probs[:, 2:].sum(axis=1)
ref_targets_bin = (targets >= 2).astype(int)
fpr, tpr, _ = roc_curve(ref_targets_bin, ref_probs)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='#1565C0', lw=2.5, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#1565C0')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title(f'ROC - Referable DR | Sens={sens:.3f}, Spec={spec:.3f}', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=12)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/validation_metrics.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved validation_metrics.png')

In [ ]:
# Final Summary
print('\n' + '=' * 50)
print('      NETRAI - FINAL RESULTS (Fold 0)')
print('=' * 50)
print(f'  Quadratic Weighted Kappa : {final_qwk:.4f}')
print(f'  Referable DR Sensitivity : {sens:.3f} ({sens*100:.1f}%)')
print(f'  Referable DR Specificity : {spec:.3f} ({spec*100:.1f}%)')
print(f'  AUC (Referable DR)       : {roc_auc:.3f}')
print('=' * 50)
print('\nDownload these files from /kaggle/working/:')
print('  netrai_fold0_best.pth   <- model weights')
print('  fold0_thresholds.npy    <- optimal class thresholds')
print('  validation_metrics.png  <- confusion matrix + ROC')
print()
for f in sorted(os.listdir('/kaggle/working/')):
    fpath = os.path.join('/kaggle/working/', f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  {f}: {size_mb:.1f} MB')